In [177]:
import json
from PIL import Image, ImageDraw, ImageFont
import subprocess

In [178]:
PROJECT_FOLDER = '/Users/sangdo/Downloads/movie-top-list/'
TRAILER_FOLDER = PROJECT_FOLDER + 'trailers/'
BG_FILE = PROJECT_FOLDER + 'timeline_template/timeline_30_items.png'
OUTPUT_LONG_VIDEO_FOLDER = PROJECT_FOLDER + 'long_video/'

first_video_x = 13
firxt_video_y = 65
CANVAS_WIDTH = 19980    #length of background image
CANVAS_HEIGHT = 1080
VIEWPORT_WIDTH = 1920
VIEWPORT_HEIGHT = 1080
VIDEO_DURATION = 240    #4 minutes
BUFFER_END_DURATION = 10    #for the last video to play to the end

CLIP_W = 640
CLIP_H = 360
#Scrollable distance: 19980 - 1920 = 18060 px
#Desired duration: 480
#Required speed: 18060 / 480 = 37.625 px/sec
SCROLL_SPEED = ((CANVAS_WIDTH - VIEWPORT_WIDTH) / VIDEO_DURATION) * 1  #pixels per sec

CARD_W = 666
#define styles of texts


<h2>Find list of trailers of the actor<h2>

In [179]:
import pymongo
import os
import subprocess

from dotenv import load_dotenv
load_dotenv(override=True) 

db_client = pymongo.MongoClient(os.environ['REMOTE_MONGO_DB'])
db = db_client['db_movie']   


In [180]:
def get_trailer_ids(actor_name):
    ids = []
    tb_paycheck = db['tb_paycheck']
    data = tb_paycheck.find({'actor': actor_name}).sort({'year':1}) #.limit(4)

    for item in data:
        ids.append(item['trailer'])
    return ids

#
# get_trailer_ids('Tom Cruise')

<h2>Put a video at position X, Y inside a video</h2>

In [182]:
def put_video_inside_video(actor_key, trailer_ids):
    inputs = ""
    filters = ""

    # build ffmpeg input list
    i = 0
    for trailer_id in trailer_ids:
        filepath = TRAILER_FOLDER + actor_key + "/"+trailer_id+".mp4"
        inputs += f"-i {filepath} "

        x = first_video_x + i * CLIP_W  #position of the clip
        trigger_x = VIEWPORT_WIDTH * 1 / 2    #start playing while reach to the center of the screen

        delay = max((x - trigger_x) / SCROLL_SPEED, 0)
        play_time = 20

        filters += (
            f"[{i+1}:v]"
            f"scale={CLIP_W}:{CLIP_H},"
            f"trim=duration={play_time},"
            f"tpad=start_duration={delay}:start_mode=clone,"
            f"tpad=stop_duration={VIDEO_DURATION + BUFFER_END_DURATION}:stop_mode=clone"
            f"[v{i}];"
        )

        i = i + 1

    # create base canvas
    filters += f"[0:v]scale={CANVAS_WIDTH}:{CANVAS_HEIGHT}[base];"

    last = "base"

    # overlay clips in position
    i = 0
    for trailer_id in trailer_ids:
        x = CARD_W * i + first_video_x
        filters += f"[{last}][v{i}]overlay={x}:{firxt_video_y}:eof_action=pass[tmp{i}];"
        last = f"tmp{i}"
        i = i + 1

    # scrolling camera
    filters += f"[{last}]crop={VIEWPORT_WIDTH}:{VIEWPORT_HEIGHT}:x='t*{SCROLL_SPEED}':y=0[out]"  #standard youtube landscape video size
    output_filename = OUTPUT_LONG_VIDEO_FOLDER + actor_key + ".mp4"
    cmd = f'ffmpeg -y -loop 1 -i {BG_FILE} {inputs} -filter_complex "{filters}" -map "[out]" -t {VIDEO_DURATION + BUFFER_END_DURATION} -r 30 -c:v libx264 -preset ultrafast -crf 23 -movflags +faststart -pix_fmt yuv420p {output_filename}'
    print("Running FFmpeg...")
    subprocess.run(cmd, shell=True)

#test
actor_name = 'Tom Cruise'
trailer_ids = get_trailer_ids(actor_name)
# print(trailer_ids)
# put_video_inside_video('tom_cruise', trailer_ids)    #size w 19980: 4 minutes ~ 1 minute